# FedDBE Client

> The core abstraction for `FedDbe` client: [https://openreview.net/forum?id=nO5i1XdUS0](https://openreview.net/forum?id=nO5i1XdUS0)

In [ ]:
#| default_exp clients.feddbe

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy
import random
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_client("feddbe")
class ClientFedDBE(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        
        self.klw = self.cfg.algorithm.kl_weight
        self.momentum = self.cfg.algorithm.momentum


### Training

In [ ]:
#| export
@patch
def detach_running(self: ClientFedDBE):
    self.running_mean.detach_()

@patch
def reset_running_stats(self: ClientFedDBE):
    self.running_mean.zero_()
    self.num_batches_tracked.zero_()

In [ ]:
#| export
@patch
def fit(self: ClientFedDBE):
    
    self.model.to(self.device)
    self.running_mean = self.running_mean.to(self.device)
    self.client_mean = self.client_mean.to(self.device)
    self.num_batches_tracked = self.num_batches_tracked.to(self.device)
    self.global_mean = self.global_mean.to(self.device) if self.global_mean is not None else None
    self.model.train()
    
    self.reset_running_stats()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            self.optimizer.zero_grad()

            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            rep = self.model.backbone(X)
            running_mean = torch.mean(rep, dim=0)

            if self.num_batches_tracked is not None:
                self.num_batches_tracked.add_(1)

            self.running_mean = (1-self.momentum) * self.running_mean + self.momentum * running_mean
            
            if self.global_mean is not None:
                reg_loss = torch.mean(0.5 * (self.running_mean - self.global_mean)**2)
                output = self.model.head(rep + self.client_mean)
                loss = self.criterion(output, y)
                loss = loss + reg_loss * self.klw
            else:
                output = self.model.head(rep)
                loss = self.criterion(output, y)
            # ====== end

            self.opt_client_mean.zero_grad()
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.opt_client_mean.step()
            self.detach_running()


### Evaluation

In [ ]:
#| export
@patch
def train_test_stats(self: ClientFedDBE, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    X, y = batch[self.data_key], batch[self.label_key]
    rep = self.model.backbone(X)
    logits = self.model.head(rep + self.client_mean)
    loss = self.criterion(logits, y)
    y_pred = logits.argmax(dim=-1)
    y_true = batch[self.label_key]

    metrics = self.training_metrics.compute(y_pred= y_pred, y_true= y_true)

    return loss, metrics


In [ ]:
#| export
@patch
def evaluate_local(self: ClientFedDBE, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []

    self.model = self.model.to(self.device)
    self.client_mean = self.client_mean.to(self.device)
    self.model.eval()
    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.train_test_stats(batch)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    self.logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    self.logger.info(f"Average {loader} Metrics: {total_metrics}")
    return {"loss": avg_loss, "metrics": total_metrics}


### Saving State

In [ ]:
#| export
@patch
def save_state(self: ClientFedDBE, save_to_disk= False):  

    state_dict = {}
    state_dict['model'] = self.model.state_dict()
    state_dict['global_mean'] = self.global_mean
    state_dict['running_mean'] = self.running_mean
    state_dict['num_batches_tracked'] = self.num_batches_tracked
    state_dict['client_mean'] = self.client_mean

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))

        torch.save(state_dict, state_path)

    return state_dict

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()